# Compilación JIT con Numba: Cuándo el Bucle Ya No Es el Enemigo

*Métodos Computacionales Modernos para la Física — Módulo II: Fundamentos de Alto Desempeño*

El cuaderno anterior (`vectorizacion_numpy.ipynb`) cerró con dos deudas
pendientes:

1. La versión vectorizada del potencial por pares gana tiempo **a cambio de
   memoria** — construye explícitamente un arreglo $O(N^2)$.
2. `numpy.vectorize` demostró, con números, que envolver un bucle no lo
   vuelve rápido; para que un bucle desaparezca de verdad hace falta que deje
   de vivir dentro del intérprete de Python.

Hay además una tercera categoría de tareas que el cuaderno anterior ni
siquiera pudo intentar vectorizar: cualquier cálculo **secuencial**, donde el
paso $n$ depende del resultado del paso $n-1$ (típicamente, integración
temporal). Ahí NumPy no tiene nada que ofrecer — no hay eje sobre el cual
hacer broadcasting.

Numba ataca las tres cosas: compila el bucle explícito a código máquina (vía
LLVM), sin exigir reescribirlo en estilo de arreglos, y por lo tanto sin pagar
el costo de memoria de la vectorización — ni sufrir su limitación de no poder
expresar una recurrencia.

Como en el cuaderno anterior: nada de lo que sigue se afirma sin medirlo.

In [1]:
import sys, math, time, tracemalloc
import numpy as np
import numba
from numba import njit, prange

print(f"Python  {sys.version.split()[0]}")
print(f"NumPy   {np.__version__}")
print(f"Numba   {numba.__version__}")
print(f"Núcleos lógicos disponibles para Numba: {numba.config.NUMBA_DEFAULT_NUM_THREADS}")

resultados = {}

Python  3.12.12
NumPy   2.4.2
Numba   0.67.0
Núcleos lógicos disponibles para Numba: 32


## 1. La mecánica de `@njit`: compilar es un costo por tipo, no por llamada

`@numba.njit` compila una función la **primera vez** que se llama con una
combinación específica de tipos de argumento — no en el momento en que se
define. Esa primera llamada paga el costo de traducir a LLVM; todas las
siguientes, con los mismos tipos, reutilizan el binario ya compilado. Antes de
tocar ningún caso de física, vale la pena ver ese costo directamente,
separando "tiempo de cómputo" de "tiempo de compilación" con un cronómetro de
una sola medición — `%timeit` promedia muchas llamadas y por diseño
escondería justo el evento que queremos ver.

In [2]:
@njit(cache=True)
def suma_cuadrados(x):
    s = 0.0
    for i in range(x.shape[0]):
        s += x[i]**2
    return s

x_demo = np.random.default_rng(0).normal(size=2_000_000)

t0 = time.perf_counter()
suma_cuadrados(x_demo)
t1 = time.perf_counter()
print(f"Primera llamada (incluye compilación): {1e3*(t1 - t0):9.2f} ms")

t0 = time.perf_counter()
suma_cuadrados(x_demo)
t1 = time.perf_counter()
print(f"Segunda llamada (ya compilada):        {1e3*(t1 - t0):9.4f} ms")

Primera llamada (incluye compilación):    167.81 ms
Segunda llamada (ya compilada):           1.0340 ms


En esta corrida la diferencia fue de más de dos órdenes de magnitud —
consistente en espíritu, aunque no en cifra exacta, con lo que la Sesión 7 ya
documentó sobre el mismo tipo de kernel (ahí la diferencia llegó a cinco
órdenes de magnitud, con un kernel más trivial y una rejilla distinta; el
número exacto depende de cuánto trabajo hace la versión ya compilada por
llamada, no solo de que exista un paso de compilación). La consecuencia
práctica para todo lo que sigue es la misma en ambos casos: cada función
`@njit` se **calienta** (se llama una vez, descartando esa medición) antes de
cronometrar su régimen estable con `%timeit`. Ignorar esto es la forma más
común de concluir, por error, que Numba "no sirvió de nada".

## 2. Caso 1 — Comprimir el overhead sin cambiar el algoritmo: Maxwell–Boltzmann

Retomamos la distribución de Maxwell–Boltzmann del cuaderno anterior,
$$f(v) = 4\pi\left(\frac{m}{2\pi k_BT}\right)^{3/2} v^2\, e^{-mv^2/2k_BT},$$
evaluada sobre una rejilla de velocidades. La versión vectorizada de NumPy ya
era rápida; la pregunta de esta sección es distinta: ¿qué tan rápido llega
**el mismo bucle explícito**, sin reescribirlo en estilo de arreglos, si en
vez de interpretarlo lo compilamos?

In [3]:
def f_mb_escalar(v, m=1.0, kT=1.0):
    return 4*math.pi*(m/(2*math.pi*kT))**1.5 * v**2 * math.exp(-m*v**2/(2*kT))

def f_mb_bucle(v_array):
    return [f_mb_escalar(v) for v in v_array]

def f_mb_vectorizada(v, m=1.0, kT=1.0):
    return 4*np.pi*(m/(2*np.pi*kT))**1.5 * v**2 * np.exp(-m*v**2/(2*kT))

@njit(cache=True)
def f_mb_njit(v, m=1.0, kT=1.0):
    n = v.shape[0]
    out = np.empty(n)
    coef = 4*np.pi*(m/(2*np.pi*kT))**1.5
    for i in range(n):
        out[i] = coef * v[i]**2 * math.exp(-m*v[i]**2/(2*kT))
    return out

v_grid = np.linspace(0.0, 10.0, 300_000)
v_grid_lista = v_grid.tolist()

f_mb_njit(v_grid[:2])  # llamada de calentamiento: compila y se descarta

assert np.allclose(f_mb_bucle(v_grid_lista), f_mb_vectorizada(v_grid))
assert np.allclose(f_mb_vectorizada(v_grid), f_mb_njit(v_grid))

In [4]:
t_mb_loop = %timeit -o f_mb_bucle(v_grid_lista)

50.8 ms ± 496 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
t_mb_vec = %timeit -o f_mb_vectorizada(v_grid)

2.28 ms ± 7.98 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [6]:
t_mb_njit = %timeit -o f_mb_njit(v_grid)

1.12 ms ± 39.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
resultados['Maxwell-Boltzmann, 3e5 puntos — bucle Python'] = t_mb_loop.average
resultados['Maxwell-Boltzmann, 3e5 puntos — NumPy vectorizado'] = t_mb_vec.average
resultados['Maxwell-Boltzmann, 3e5 puntos — @njit'] = t_mb_njit.average

print(f"@njit vs. bucle Python : {t_mb_loop.average / t_mb_njit.average:6.0f}x")
print(f"@njit vs. NumPy        : {t_mb_vec.average  / t_mb_njit.average:6.2f}x")

@njit vs. bucle Python :     45x
@njit vs. NumPy        :   2.03x


No hicimos falta broadcasting ni `ufunc`: el mismo `for` de siempre,
compilado, alcanza (o supera) a la versión vectorizada de NumPy — porque el
bucle compilado por LLVM y el bucle en C que corre dentro de una ufunc de
NumPy son, en esencia, el mismo tipo de código máquina. La ventaja de Numba
aquí no es de velocidad pura sobre NumPy; es que **no exigió cambiar de estilo
de programación** para conseguirla.

## 3. Caso 2 — La deuda de memoria pendiente: potencial por pares

El cuaderno anterior dejó una advertencia explícita sobre el potencial
gravitacional por pares,
$$U = -G\sum_{i<j}\frac{m_im_j}{r_{ij}},$$
vectorizado con `pos[:, None, :] - pos[None, :, :]`: esa construcción es
rápida, pero es $O(N^2)$ **en memoria**, no solo en tiempo. Un bucle doble
compilado con `@njit` resuelve ambos frentes a la vez — seguimos haciendo
$O(N^2)$ operaciones (no hay manera de evitar contar cada par), pero sin
materializar nunca un arreglo de tamaño $N^2$.

Usamos aquí $N$ más grande que en el cuaderno anterior para que la diferencia
de memoria dependas de una cifra real, no de una promesa asintótica.

In [8]:
rng = np.random.default_rng(42)
N = 1500
pos = rng.uniform(-1.0, 1.0, size=(N, 3))
masas = rng.uniform(0.5, 2.0, size=N)
G = 1.0

def potencial_bucle(pos, masas, G):
    N = len(masas)
    U = 0.0
    for i in range(N):
        for j in range(i+1, N):
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]
            r = math.sqrt(dx*dx + dy*dy + dz*dz)
            U -= G*masas[i]*masas[j]/r
    return U

def potencial_broadcasting(pos, masas, G):
    diff = pos[:, None, :] - pos[None, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1))
    np.fill_diagonal(r, np.inf)
    U_pares = -G * np.outer(masas, masas) / r
    return 0.5 * np.sum(U_pares)

@njit(cache=True)
def potencial_njit(pos, masas, G):
    N = masas.shape[0]
    U = 0.0
    for i in range(N):
        for j in range(i+1, N):
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]
            r = math.sqrt(dx*dx + dy*dy + dz*dz)
            U -= G*masas[i]*masas[j]/r
    return U

potencial_njit(pos[:2], masas[:2], G)  # calentamiento

u_bucle = potencial_bucle(pos, masas, G)
u_bcast = potencial_broadcasting(pos, masas, G)
u_njit = potencial_njit(pos, masas, G)
assert np.isclose(u_bucle, u_bcast)
assert np.isclose(u_bucle, u_njit)

In [9]:
# La versión con bucles de Python es lenta: pocas repeticiones.
t_pot_loop = %timeit -o -n 1 -r 3 potencial_bucle(pos, masas, G)

647 ms ± 1.15 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [10]:
t_pot_bcast = %timeit -o potencial_broadcasting(pos, masas, G)

50.4 ms ± 61.5 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
t_pot_njit = %timeit -o potencial_njit(pos, masas, G)

2.17 ms ± 3.95 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [12]:
resultados[f'Potencial por pares, N={N} — bucle Python'] = t_pot_loop.average
resultados[f'Potencial por pares, N={N} — NumPy (broadcasting)'] = t_pot_bcast.average
resultados[f'Potencial por pares, N={N} — @njit'] = t_pot_njit.average

print(f"@njit vs. bucle Python      : {t_pot_loop.average  / t_pot_njit.average:6.0f}x")
print(f"@njit vs. NumPy broadcasting: {t_pot_bcast.average / t_pot_njit.average:6.2f}x")

@njit vs. bucle Python      :    298x
@njit vs. NumPy broadcasting:  23.22x


Tiempo aparte, la razón por la que esta sección existe es la memoria.
Medimos el pico real de memoria — no lo calculamos a mano — con
`tracemalloc`, para las dos versiones que sí devuelven el resultado correcto
en tiempo competitivo.

In [13]:
tracemalloc.start()
potencial_broadcasting(pos, masas, G)
_, pico_bcast = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
potencial_njit(pos, masas, G)
_, pico_njit = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Pico de memoria — NumPy (broadcasting): {pico_bcast/1e6:7.2f} MB")
print(f"Pico de memoria — @njit (bucle)       : {pico_njit/1e6:7.2f} MB")
print(f"Razón                                 : {pico_bcast/max(pico_njit,1):7.0f}x")

Pico de memoria — NumPy (broadcasting):  126.00 MB
Pico de memoria — @njit (bucle)       :    0.02 MB
Razón                                 :    6484x


El arreglo `(N, N, 3)` que construye `pos[:, None, :] - pos[None, :,
:]` domina el consumo de memoria de la versión vectorizada; el bucle
compilado, en cambio, solo necesita un puñado de variables escalares
reutilizadas $N^2/2$ veces. Esto es exactamente lo que el cuaderno anterior
dejó pendiente: Numba no solo iguala a NumPy en velocidad aquí — lo hace sin
pagar el costo de memoria que hacía a la versión vectorizada, en el límite de
$N$ grande, insostenible.

## 4. Caso 3 — Donde NumPy no tiene nada que ofrecer: integración temporal

Los dos casos anteriores tenían una alternativa vectorizada de NumPy contra
la cual comparar. Esta no la tiene, y por una razón estructural: vamos a
integrar la trayectoria de un péndulo forzado y amortiguado,
$$\ddot\theta + \gamma\dot\theta + \frac{g}{L}\sin\theta = A\cos(\Omega t),$$
con el método de Runge–Kutta de cuarto orden. Cada paso de RK4 necesita el
estado $(\theta,\omega)$ que dejó el paso anterior — es una **recurrencia**,
no una operación elemento a elemento sobre una rejilla. No existe un eje
sobre el cual aplicar broadcasting: no hay "versión NumPy" de este algoritmo
que no sea, en el fondo, el mismo bucle secuencial. Aquí Numba no compite con
NumPy; compite únicamente con el intérprete de Python.

In [14]:
def deriv(theta, omega, t, g_L, gamma, A, Omega):
    dtheta = omega
    domega = -g_L*math.sin(theta) - gamma*omega + A*math.cos(Omega*t)
    return dtheta, domega

def integrar_bucle(theta0, omega0, dt, n_pasos, g_L, gamma, A, Omega):
    theta, omega, t = theta0, omega0, 0.0
    thetas = np.empty(n_pasos)
    for k in range(n_pasos):
        k1t, k1o = deriv(theta, omega, t, g_L, gamma, A, Omega)
        k2t, k2o = deriv(theta + 0.5*dt*k1t, omega + 0.5*dt*k1o, t + 0.5*dt, g_L, gamma, A, Omega)
        k3t, k3o = deriv(theta + 0.5*dt*k2t, omega + 0.5*dt*k2o, t + 0.5*dt, g_L, gamma, A, Omega)
        k4t, k4o = deriv(theta + dt*k3t, omega + dt*k3o, t + dt, g_L, gamma, A, Omega)
        theta += (dt/6.0)*(k1t + 2*k2t + 2*k3t + k4t)
        omega += (dt/6.0)*(k1o + 2*k2o + 2*k3o + k4o)
        t += dt
        thetas[k] = theta
    return thetas

@njit(cache=True)
def deriv_njit(theta, omega, t, g_L, gamma, A, Omega):
    dtheta = omega
    domega = -g_L*math.sin(theta) - gamma*omega + A*math.cos(Omega*t)
    return dtheta, domega

@njit(cache=True)
def integrar_njit(theta0, omega0, dt, n_pasos, g_L, gamma, A, Omega):
    theta, omega, t = theta0, omega0, 0.0
    thetas = np.empty(n_pasos)
    for k in range(n_pasos):
        k1t, k1o = deriv_njit(theta, omega, t, g_L, gamma, A, Omega)
        k2t, k2o = deriv_njit(theta + 0.5*dt*k1t, omega + 0.5*dt*k1o, t + 0.5*dt, g_L, gamma, A, Omega)
        k3t, k3o = deriv_njit(theta + 0.5*dt*k2t, omega + 0.5*dt*k2o, t + 0.5*dt, g_L, gamma, A, Omega)
        k4t, k4o = deriv_njit(theta + dt*k3t, omega + dt*k3o, t + dt, g_L, gamma, A, Omega)
        theta += (dt/6.0)*(k1t + 2*k2t + 2*k3t + k4t)
        omega += (dt/6.0)*(k1o + 2*k2o + 2*k3o + k4o)
        t += dt
        thetas[k] = theta
    return thetas

parametros = dict(theta0=2.0, omega0=0.0, dt=1e-3, n_pasos=500_000,
                   g_L=1.0, gamma=0.2, A=0.5, Omega=0.6666)

integrar_njit(2.0, 0.0, 1e-3, 10, 1.0, 0.2, 0.5, 0.6666)  # calentamiento

theta_bucle = integrar_bucle(**parametros)
theta_njit = integrar_njit(*parametros.values())
assert np.allclose(theta_bucle, theta_njit)

In [15]:
t_pend_loop = %timeit -o -n 1 -r 3 integrar_bucle(**parametros)

290 ms ± 273 μs per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [16]:
t_pend_njit = %timeit -o integrar_njit(*parametros.values())

20.6 ms ± 26.7 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [17]:
resultados[f'Péndulo RK4, {parametros["n_pasos"]:,} pasos — bucle Python'] = t_pend_loop.average
resultados[f'Péndulo RK4, {parametros["n_pasos"]:,} pasos — @njit'] = t_pend_njit.average

print(f"Ganancia: {t_pend_loop.average / t_pend_njit.average:,.0f}x")
print(f"Tiempo por paso de RK4 — bucle Python: {1e9*t_pend_loop.average/parametros['n_pasos']:6.1f} ns")
print(f"Tiempo por paso de RK4 — @njit       : {1e9*t_pend_njit.average/parametros['n_pasos']:6.1f} ns")

Ganancia: 14x
Tiempo por paso de RK4 — bucle Python:  579.7 ns
Tiempo por paso de RK4 — @njit       :   41.2 ns


Esta es, de las tres, la ganancia más limpia conceptualmente: no hubo
que elegir entre dos algoritmos ni pagar memoria adicional — el bucle
secuencial de RK4 es exactamente el mismo en ambas versiones, línea por
línea. La única diferencia es si cada una de sus operaciones aritméticas
cruza o no la frontera del intérprete de Python 2 millones de veces
($4\times$ pasos, por las cuatro evaluaciones de `deriv` en cada paso de
RK4).

## 5. Bonus — `parallel=True` y `prange`: más allá de un solo núcleo

Numba puede repartir un bucle `@njit` entre varios hilos con solo declarar
`parallel=True` y cambiar `range` por `prange` en el índice que se quiere
paralelizar — sin escribir código de sincronización a mano. Lo probamos sobre
el potencial por pares del Caso 2, en esta máquina con
32 núcleos lógicos disponibles.

Nota de implementación: `prange` reparte bien un bucle **rectangular**; la
versión anterior recorría solo $j>i$ (triangular) para no contar cada par dos
veces. Aquí recorremos la matriz completa $N\times N$ (contando cada par dos
veces y dividiendo entre 2 al final, igual que la versión de broadcasting)
porque reparte el trabajo de forma uniforme entre hilos — la comparación justa
no es "menos trabajo total", es "el mismo trabajo, en paralelo".

In [18]:
@njit(parallel=True, cache=True)
def potencial_njit_paralelo(pos, masas, G):
    N = masas.shape[0]
    total = 0.0
    for i in prange(N):
        for j in range(N):
            if i != j:
                dx = pos[i, 0] - pos[j, 0]
                dy = pos[i, 1] - pos[j, 1]
                dz = pos[i, 2] - pos[j, 2]
                r = math.sqrt(dx*dx + dy*dy + dz*dz)
                total -= G*masas[i]*masas[j]/r
    return 0.5*total

potencial_njit_paralelo(pos[:2], masas[:2], G)  # calentamiento
assert np.isclose(potencial_njit_paralelo(pos, masas, G), u_bucle)

In [19]:
t_pot_par = %timeit -o potencial_njit_paralelo(pos, masas, G)

resultados[f'Potencial por pares, N={N} — @njit (parallel, {numba.config.NUMBA_DEFAULT_NUM_THREADS} hilos)'] = t_pot_par.average
print(f"@njit paralelo vs. @njit secuencial: {t_pot_njit.average / t_pot_par.average:5.2f}x")

737 μs ± 22.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
@njit paralelo vs. @njit secuencial:  2.94x


La ganancia real del paralelismo casi nunca es igual al número de
núcleos — hay overhead de repartir y recolectar el trabajo entre hilos, y con
$N=1500$ cada hilo recibe una porción todavía modesta. Vale como demostración
de que la puerta existe; explotarla a fondo (granularidad, *false sharing*,
cuándo el overhead de lanzar hilos supera la ganancia) es, con toda
intención, el tema de la siguiente unidad del curso —
**Paralelismo de Memoria Compartida** — no de esta sección.

## 6. Resumen de lo medido

Igual que en el cuaderno anterior: nada de esta tabla está escrito a mano.

In [20]:
ancho = max(len(k) for k in resultados)
print(f"{'Experimento':<{ancho}}  {'t promedio':>14}")
print('-' * (ancho + 16))
for nombre, t in resultados.items():
    if t < 1e-3:
        texto = f"{t*1e6:9.1f} µs"
    elif t < 1.0:
        texto = f"{t*1e3:9.2f} ms"
    else:
        texto = f"{t:9.3f} s "
    print(f"{nombre:<{ancho}}  {texto:>14}")

Experimento                                                   t promedio
------------------------------------------------------------------------
Maxwell-Boltzmann, 3e5 puntos — bucle Python                    50.82 ms
Maxwell-Boltzmann, 3e5 puntos — NumPy vectorizado                2.28 ms
Maxwell-Boltzmann, 3e5 puntos — @njit                            1.12 ms
Potencial por pares, N=1500 — bucle Python                     646.71 ms
Potencial por pares, N=1500 — NumPy (broadcasting)              50.35 ms
Potencial por pares, N=1500 — @njit                              2.17 ms
Péndulo RK4, 500,000 pasos — bucle Python                      289.86 ms
Péndulo RK4, 500,000 pasos — @njit                              20.59 ms
Potencial por pares, N=1500 — @njit (parallel, 32 hilos)        736.7 µs


## 7. Para llevar

* Compilar no cambia el algoritmo. En los Casos 1 y 3, el `@njit` es
  literalmente el mismo código que la versión "lenta" con un decorador
  encima — la ganancia viene enteramente de que el bucle deja de vivir en el
  intérprete de Python.
* Numba resuelve el compromiso de memoria que NumPy no puede: el Caso 2 logra
  la velocidad de la versión vectorizada **sin** el arreglo $O(N^2)$ que esa
  versión necesita — medido con `tracemalloc`, no supuesto.
* Hay una categoría entera de tareas de física — cualquier recurrencia
  temporal, el Caso 3 de este cuaderno — donde NumPy vectorizado simplemente
  no aplica. Ahí Numba no es "una alternativa más rápida"; es la única
  herramienta, de las que ha visto este curso hasta ahora, que funciona.
* El costo de compilación es real pero se paga una sola vez por firma de
  tipos (Sección 1) — hay que calentar antes de medir, y hay que amortizar
  ese costo sobre suficientes llamadas para que valga la pena.
* `parallel=True`/`prange` (Sección 5) abre la puerta a varios núcleos casi
  gratis en líneas de código, aunque la ganancia real depende de la
  granularidad del trabajo — el tema completo de la siguiente unidad.
* Lo que sigue sin resolverse — el proyecto del semestre necesita explorar
  miles o millones de combinaciones de parámetros, no solo acelerar una
  evaluación — es exactamente la motivación de **MPI** y el cómputo
  distribuido, más adelante en el Módulo II.